# 04.2 · kv-transfer (Spec-2) — data pipeline demo

`SRC.task != DST.task`, `SRC.progress == DST.progress`. Two distinct tasks that share a **common progress** (a prerequisite sub-computation) are clustered together; both prefill the same progress code. Transplant the SRC **task-description key+value** onto DST (GQA-corrected headwise routing) → does DST implement SRC's task?

Also demonstrates **cross-language** transfer: cache the task in Python, trigger it in C++.

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "2, 3")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

sys.path.append("../..")  # repo root, so `src` is importable
import torch, numpy as np
import src.diff.functional as P
from src.diff.functional import *          # helpers into scope
import src.diff.data as D                   # dataset generators

P.setup("Qwen/Qwen2.5-Coder-32B-Instruct") # loads model + head masks (once)
mt = P.mt
GOAL_HEADS, BOUNDARY_HEADS, UNION_HEADS, GROUP = P.GOAL_HEADS, P.BOUNDARY_HEADS, P.UNION_HEADS, P.GROUP
from rich.panel import Panel
from rich.text import Text
from rich.markup import escape

def pivot_of(prompt, neg_shift=-1):
    return prepare_input(prompts=prompt, tokenizer=mt)["input_ids"].shape[1] + neg_shift

def show(prompt, title, emph_span=None, piv=-1):
    kw = dict(patch_pos=[piv])
    if emph_span is not None:
        kw["emphasize_tokens"] = range(*emph_span)
    rich_print(Panel(Text.from_markup(exact_prompt(prompt, **kw)),
                     title=f"[bold]{title}[/bold]", title_align="left",
                     border_style="grey42", padding=(1, 1)))

2026-09-05 19:45:10 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

Qwen/Qwen2.5-Coder-32B-Instruct
n_heads=40  n_kv_heads=8  group=5  head_dim=128  n_layer=64
GOAL_HEADS     : 122  (layers 28–55)
BOUNDARY_HEADS : 140  (layers 11–62)
UNION_HEADS    : 228


## Intervention knobs

`ALPHA` — OV-write boost of `PATCH_HEADS` at the pivot during the **patched** gen/readout (`1.0` = off) — the only knob; every intervention here is a **span** k+v transplant (no pivot-query window). Default is a no-op.

In [2]:
from contextlib import nullcontext

#------------------------------------------------------------------------------------------------------
ALPHA = 2.0        # OV-write boost of PATCH_HEADS at the pivot on the PATCHED gen/readout (1.0 = off)
ALIGN_KNOB = "pad"   # k/v transplant window (A4 knob): "right" = right-edge align (drop the longer
                       #   span's front) ; "pad" = left-pad the shorter span so the dest span is fully
                       #   overwritten (works cross-language too)
PATCH_HEADS = GOAL_HEADS                    # heads carrying the transplant (GOAL | UNION | ...)
READOUT_HEADS = GOAL_HEADS                  # heads read by goalscope (I3); kept SEPARATE from PATCH_HEADS
# ── ablation (I2): ZERO the OV write of these heads on the PATCHED run — kill the "backup" goal
# calc the KV-patch inconsistency triggers. Default = every head in ABLATE_LAYERS minus PATCH_HEADS.
ABLATE_LAYERS = range(33, 43)   # layers 35–50 (inclusive) — where GOAL_HEADS concentrate
ABLATE_HEADS = sorted({(l, h) for l in ABLATE_LAYERS for h in range(mt.n_heads)} - set(PATCH_HEADS))
# ABLATE_HEADS = []
ABLATION_STRAT = "restrict"   # how ABLATE_HEADS are ablated on the PATCHED run:
#  "zero"     -> kill their OV write (×0.0 at ALL positions) -- the original strategy
#  "restrict" -> keep their write but attention-knockout the <task_span>: they attend to every
#                context token EXCEPT the task-description columns (pre-softmax -inf)
#------------------------------------------------------------------------------------------------------

def ov_boost(heads=None, alpha=None, positions=(-1,)):
    """Context manager scaling the OV write of `heads` by `alpha` (defaults PATCH_HEADS/ALPHA);
    alpha==1.0 -> nullcontext (no-op). Stack it under `headwise_kv_patch` for the patched gen."""
    heads = PATCH_HEADS if heads is None else heads
    alpha = ALPHA if alpha is None else alpha
    return boost_head_ov(mt, heads, alpha, positions=positions) if alpha != 1.0 else nullcontext()



def ablate_ctx(heads=None, keep=None, block_span=None):
    """Ablate `heads` (default ABLATE_HEADS) on the PATCHED run; `keep` heads (default PATCH_HEADS)
    are never ablated so a patched head is never touched. Strategy per ABLATION_STRAT:
      "zero"     -> kill their OV write (boost_head_ov x0.0 at ALL positions);
      "restrict" -> attention-knockout: block their attention to `block_span` (the DST <task_span>)
                    so they can look everywhere else. "restrict" requires `block_span=(start, end)`."""
    heads = ABLATE_HEADS if heads is None else heads
    keep = set(PATCH_HEADS if keep is None else keep)
    heads = [h for h in heads if h not in keep]
    if not heads:
        return nullcontext()
    if ABLATION_STRAT == "restrict":
        assert block_span is not None, "restrict ablation needs block_span (the DST task span)"
        return restrict_head_attn(mt, heads, range(block_span[0], block_span[1]), keep=keep)
    return boost_head_ov(mt, heads, 0.0, positions=None)

def headwise_iface(patch_heads, k_cache, v_cache, dst_pos, alpha=None):
    """headwise_kv_patch patch_interface, composed with the ALPHA OV-boost — for the PATCHED
    goalscope readout. At ALPHA==1.0 it matches the plain `make_headwise_interface` iface."""
    def _if(mt, inputs, patches=None, model_kwargs=None):
        with headwise_kv_patch(mt, patch_heads, k_cache, v_cache, dst_pos), ov_boost(patch_heads, alpha):
            return mt._model(**dict(inputs), **dict(model_kwargs or {}))
    return _if


# ── two-span prompt renderer: full task span (bright_yellow) + the EXACT KV window (orange) ────
# The transplant window is chosen by `align_kv_spans(mt, ..., mode=ALIGN_KNOB)` (src.diff.functional).
def show_spans(tok, title, task_span=None, kv_span=None, piv=-1):
    """Render the prompt exactly as the model sees it. task_span -> bright_yellow (the task text);
    kv_span -> `black on dark_orange` (the exact tokens whose k+v are transplanted); pivot -> salmon.
    Priority pivot > kv > task, so the KV window shows through the task highlight."""
    ids = (prepare_input(prompts=tok, tokenizer=mt) if isinstance(tok, str) else tok)["input_ids"][0].tolist()
    n = len(ids)
    pivpos = n + piv if piv < 0 else piv
    task = set(range(*task_span)) if task_span else set()
    kv = set(range(*kv_span)) if kv_span else set()
    # Mark spans with control-char sentinels and escape the WHOLE joined string once (then swap
    # sentinels for tags). Escaping per-token drops brackets like `[w ...]` / `list[str]` when the
    # tokenizer splits `[` from the following letter across two tokens: the lone `[` isn't escaped,
    # so once joined it reads as a markup tag and rich absorbs it.
    PO, PC, KO, KC, TO, TC = "\x00<", "\x00>", "\x01<", "\x01>", "\x02<", "\x02>"
    parts = []
    for i in range(n):
        s = mt.tokenizer.decode([ids[i]]).replace("\n", "\\n\n")
        if i == pivpos:
            s = PO + s + PC
        elif i in kv:
            s = KO + s + KC
        elif i in task:
            s = TO + s + TC
        parts.append(s)
    txt = escape("".join(parts))
    txt = (txt.replace(PO, "[bold black on salmon1]").replace(PC, "[/bold black on salmon1]")
              .replace(KO, "[bold black on dark_orange]").replace(KC, "[/bold black on dark_orange]")
              .replace(TO, "[bright_yellow]").replace(TC, "[/bright_yellow]"))
    rich_print(Panel(Text.from_markup(txt), title=f"[bold]{escape(title)}[/bold]",
                     title_align="left", border_style="grey42", padding=(1, 1)))

## Inspect the generated clusters / setups

In [3]:
for dt, clusters in D.KVCLUSTERS.items():
    print(f"{dt}:")
    for cl in clusters:
        print(f"  cluster {cl.id!r}: prereq={cl.common_progress.result_name!r} "
              f"({cl.common_progress.description!r}) · tasks={[t.id for t in cl.tasks]}")
print()
setups = D.kv_transfer_setups("list[str]", presentation="description")
print(f"list[str]: {len(setups)} ordered pairs across its clusters")
for s in setups[:12]:
    print(f"  [{s.meta['cluster']:<14}] src={s.src_task:<22} dst={s.dst_task:<22} pivot={s.pivot}")

list[str]:
  cluster 'sorted_by_len': prereq='sorted_words' ('the words sorted by length') · tasks=['shortest_vowel_start', 'longest_palindrome', 'longest_len', 'shortest_len', 'median_word_len', 'longest_starts_upper']
  cluster 'long_words': prereq='long_words' ('the words that have at least two characters') · tasks=['count_palindromes', 'count_all_distinct', 'count_same_ends']
  cluster 'lowercased': prereq='lowered' ('the words converted to lowercase') · tasks=['count_ci_palindromes', 'num_distinct_ci', 'count_contains_a', 'any_ci_duplicate']
list[int]:
  cluster 'sorted_asc': prereq='sorted_nums' ('the values sorted in ascending order') · tasks=['median_val', 'range_span', 'second_largest', 'second_smallest', 'max_gap']
  cluster 'positives': prereq='positives' ('the values that are greater than zero') · tasks=['sum_positives', 'count_positives', 'product_positives', 'smallest_positive', 'all_even_positives']
  cluster 'prefix_sums': prereq='prefix' ('the running cumulative sums o

## Render one setup — SRC/DST prompts with the transplanted task span marked

In [4]:
inst = D.kv_transfer_setups(
    "list[str]", presentation="description", src_lang="python", dst_lang="python",
)[4]
m = D.materialize(inst)
SRC_SPAN0 = token_span(m["src_prompt"], m["src_task_text"])   # located target spans (pre-align)
DST_SPAN0 = token_span(m["dst_prompt"], m["dst_task_text"])
print(f"SRC task = {m['src_task_text']!r}\nDST task = {m['dst_task_text']!r}")
# align -> TokenizerOutputs (pad token-splices \n before the shorter target). SRC_POS = cached-from,
# DST_POS = patched-to. For pad both spans are K=max -> the whole span is orange (fully patched).
src_tok, dst_tok, SRC_POS, DST_POS, SRC_SPAN, DST_SPAN = align_kv_spans(
    mt, m["src_prompt"], m["dst_prompt"], SRC_SPAN0, DST_SPAN0, mode=ALIGN_KNOB,
    src_text=m["src_task_text"], dst_text=m["dst_task_text"])
piv_src, piv_dst = src_tok["input_ids"].shape[1] - 1, dst_tok["input_ids"].shape[1] - 1
show_spans(src_tok, f"SRC · CACHED-FROM {tuple(SRC_SPAN)} (orange) · pivot@{piv_src} · align={ALIGN_KNOB}",
           task_span=SRC_SPAN, kv_span=(SRC_POS[0], SRC_POS[-1] + 1), piv=-1)
show_spans(dst_tok, f"DST · PATCHED-TO {tuple(DST_SPAN)} (orange) · pivot@{piv_dst} · align={ALIGN_KNOB}",
           task_span=DST_SPAN, kv_span=(DST_POS[0], DST_POS[-1] + 1), piv=-1)

SRC task = 'the length of the median word when the words are ordered by length'
DST task = 'the length of the shortest word'


╭─ SRC · CACHED-FROM (63, 76) (orange) · pivot@103 · align=pad ───────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ the length of the median word when the words are ordered by length\n                                            │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     sorted_words = sorted(words, key=len)\n                                                                     │
│     return                                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ DST · PATCHED-TO (63, 76) (orange) · pivot@103 · align=pad ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ \n                                                                                                              │
│ \n                                                                                                              │
│ \n                                                                                                              │
│ \n                                                                                                              │
│ \n                                                                                                              │
│ \n                                                                                                              │
│ \n                                                                                                              │
│ the length of the shortest word\n                                                                               │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     sorted_words = sorted(words, key=len)\n                                                                     │
│     return                                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Intervention — transplant SRC task-description key+value onto DST

Align the two task spans (per `ALIGN_KNOB` — right-edge or pad), cache SRC key+value over `PATCH_HEADS`, and run the destination under `headwise_kv_patch` (only `PATCH_HEADS` read the source span — GQA bleed corrected). Expect patched DST to implement SRC's task.

In [5]:
# ── intervention: transplant SRC task-description k+v onto DST — grouped per run ──
from src.diff.functional import _layers_of, heads_per_layer, cache_src_proj

# ALPHA = 3.0
# PATCH_HEADS = UNION_HEADS

# src_tok / dst_tok / SRC_POS (cached-from) / DST_POS (patched-to) / piv come from the render cell above
kvheads = heads_per_layer(PATCH_HEADS, "k", layers=set(_layers_of(PATCH_HEADS)))
k_cache = cache_src_proj(src_tok, SRC_POS, sorted(kvheads), "k")
v_cache = cache_src_proj(src_tok, SRC_POS, sorted(kvheads), "v")

# ── source_run — clean SRC ──
gen_src = greedy(src_tok, max_new=64)
rich_print(f"[red1]source_run · clean SRC ({m['src_task_text']!r}):[/red1] \n{escape(gen_src)}")
ro_src = goalscope_readout("   ↳ goalscope · source_run", src_tok, piv_src, heads=READOUT_HEADS)[0]["patched_gen"]

# ── destination_run — clean DST ──
gen_dst = greedy(dst_tok, max_new=64)
rich_print(f"[sea_green3]destination_run · clean DST ({m['dst_task_text']!r}):[/sea_green3] \n{escape(gen_dst)}")
ro_dst = goalscope_readout("   ↳ goalscope · destination_run", dst_tok, piv_dst, heads=READOUT_HEADS)[0]["patched_gen"]

# ── patched_run — SRC task k+v transplanted (+ ALPHA OV-boost) ──
_boost = boost_head_ov(mt, PATCH_HEADS, ALPHA, positions=(-1,)) if ALPHA != 1.0 else nullcontext()
with headwise_kv_patch(mt, PATCH_HEADS, k_cache, v_cache, DST_POS), _boost, ablate_ctx(keep=PATCH_HEADS, block_span=DST_SPAN):
    _patched = greedy(dst_tok, max_new=64)
rich_print(f"[salmon1]patched_run · DST (expect {m['src_task_text']!r}):[/salmon1] \n{escape(_patched)}")
ro_patched = goalscope_readout("   ↳ goalscope · patched_run", dst_tok, piv_dst, heads=READOUT_HEADS,
                  iface=headwise_iface(PATCH_HEADS, k_cache, v_cache, DST_POS))[0]["patched_gen"]

# ── eval pipeline — causal + readout (no gaze) ──
from src.diff.eval import eval_kv_transfer
_ev = eval_kv_transfer(inst, patched_gen=_patched, readouts=(ro_src, ro_dst, ro_patched))
rich_print(f"[bold]eval · kv-cluster[/bold] : {_ev.summary()}")

source_run · clean SRC ('the length of the median word when the words are ordered by length'): 
 len(sorted_words[len(sorted_words) // 2])
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


   ↳ goalscope · source_run  pos=103 ' return' => I'm calculating the length of the middle element in a sorted list
of strings, where the list is sorted by the length of the strings.

destination_run · clean DST ('the length of the shortest word'): 
 len(sorted_words[0])
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · destination_run  pos=103 ' return' => I'm determining the length of the shortest string in the 
list by using the `len()` function on the first element after sorting the list by string length.

patched_run · DST (expect 'the length of the median word when the words are ordered by length'): 
 len(sorted_words[len(sorted_words) // 2])
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · patched_run  pos=103 ' return' => I'm calculating the length of the middle element in a sorted 
list of strings, where the list is sorted by string length.

eval · kv-cluster : causal: TRIGGERED (src) | readout: src→dst→src

## Cross-language — cache the task in Python, trigger it in a C++ destination

In [6]:
# ── cross-language: cache the task in Python, trigger it in a C++ destination — grouped per run ──
xinst = D.kv_transfer_setups(
    "list[str]", presentation="description", src_lang="python", dst_lang="cpp"
)[3]
xm = D.materialize(xinst)
XS0 = token_span(xm["src_prompt"], xm["src_task_text"])
XD0 = token_span(xm["dst_prompt"], xm["dst_task_text"])
xsrc_tok, xdst_tok, XSP, XDP, XS, XD = align_kv_spans(
    mt, xm["src_prompt"], xm["dst_prompt"], XS0, XD0, mode=ALIGN_KNOB,
    src_text=xm["src_task_text"], dst_text=xm["dst_task_text"])
xpiv_src, xpiv_dst = xsrc_tok["input_ids"].shape[1] - 1, xdst_tok["input_ids"].shape[1] - 1
show_spans(xsrc_tok, f"SRC (python) · CACHED-FROM {tuple(XS)} (orange) · align={ALIGN_KNOB}",
           task_span=XS, kv_span=(XSP[0], XSP[-1] + 1), piv=-1)
show_spans(xdst_tok, f"DST (c++)    · PATCHED-TO {tuple(XD)} (orange) · align={ALIGN_KNOB}",
           task_span=XD, kv_span=(XDP[0], XDP[-1] + 1), piv=-1)

xkvh = heads_per_layer(PATCH_HEADS, "k", layers=set(_layers_of(PATCH_HEADS)))
xk = cache_src_proj(xsrc_tok, XSP, sorted(xkvh), "k")
xv = cache_src_proj(xsrc_tok, XSP, sorted(xkvh), "v")

# ── source_run (python) ──
xgen_src = greedy(xsrc_tok, max_new=64)
rich_print(f"[red1]source_run · clean SRC/python ({xm['src_task_text']!r}):[/red1] \n{escape(xgen_src)}")
xro_src = goalscope_readout("   ↳ goalscope · source_run", xsrc_tok, xpiv_src, heads=READOUT_HEADS)[0]["patched_gen"]

# ── destination_run (c++) ──
xgen_dst = greedy(xdst_tok, max_new=64)
rich_print(f"[sea_green3]destination_run · clean DST/c++ ({xm['dst_task_text']!r}):[/sea_green3] \n{escape(xgen_dst)}")
xro_dst = goalscope_readout("   ↳ goalscope · destination_run", xdst_tok, xpiv_dst, heads=READOUT_HEADS)[0]["patched_gen"]

# ── patched_run — python task k+v -> c++ DST (expect the PYTHON src task) ──
_xboost = boost_head_ov(mt, PATCH_HEADS, ALPHA, positions=(-1,)) if ALPHA != 1.0 else nullcontext()
with headwise_kv_patch(mt, PATCH_HEADS, xk, xv, XDP), _xboost, ablate_ctx(keep=PATCH_HEADS, block_span=XD):
    xpatched = greedy(xdst_tok, max_new=64)
rich_print(f"[salmon1]patched_run · DST/c++ (expect {xm['src_task_text']!r}):[/salmon1] \n{escape(xpatched)}")
xro_patched = goalscope_readout("   ↳ goalscope · patched_run", xdst_tok, xpiv_dst, heads=READOUT_HEADS,
                  iface=headwise_iface(PATCH_HEADS, xk, xv, XDP))[0]["patched_gen"]

from src.diff.eval import eval_kv_transfer
_xev = eval_kv_transfer(xinst, patched_gen=xpatched, readouts=(xro_src, xro_dst, xro_patched))
rich_print(f"[bold]eval · kv-cluster (cross-lang)[/bold] : {_xev.summary()}")

╭─ SRC (python) · CACHED-FROM (63, 73) (orange) · align=pad ──────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ returns true if the shortest word starts with a vowel\n                                                         │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     sorted_words = sorted(words, key=len)\n                                                                     │
│     return                                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ DST (c++)    · PATCHED-TO (64, 74) (orange) · align=pad ───────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a C++ program that implements a function `func` which, given a list of strings `words`, computes:\n       │
│ \n                                                                                                              │
│ returns true if the longest word is a palindrome\n                                                              │
│ Assume that all the libraries have been already imported.\n                                                     │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ auto func(std::vector<std::string> words){\n                                                                    │
│     std::vector<std::string> sorted_words = words;\n                                                            │
│     std::stable_sort(sorted_words.begin(), sorted_words.end(), [](const std::string& a, const std::string& b){  │
│ return a.size() < b.size(); });\n                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

source_run · clean SRC/python ('returns true if the shortest word starts with a vowel'): 
 sorted_words[0][0].lower() in 'aeiou'
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · source_run  pos=100 ' return' => I'm checking if the first character of a string is a vowel.

destination_run · clean DST/c++ ('returns true if the longest word is a palindrome'): 
 std::string longest_word = sorted_words.back();
    std::string reversed_word = longest_word;
    std::reverse(reversed_word.begin(), reversed_word.end());
    return longest_word == reversed_word;
}
</code>

... (2 outputs left out)

patched_run · DST/c++ (expect 'returns true if the shortest word starts with a vowel'): 
 return !sorted_words.empty() && "aeiouAEIOU".find(sorted_words[0][0]) != std::string::npos;
}
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

   ↳ goalscope · patched_run  pos=155 '    ' => I'm checking if the first character of the first string in a list 
is a vowel.

eval · kv-cluster (cross-lang) : causal: TRIGGERED (src) | readout: src→dst→src

## Multi-predicate setup — `SRC(T1,T2,T3)` vs `DST(T1,T2,T4)` (kv-transfer)

A kv-transfer variant purpose-built for this notebook. SRC and DST share a **prefilled prefix** (T1, T2 assigned identically) and diverge only in the **frontier** predicate their *question* lists — SRC asks for T3, DST for T4. The frontier description is a distinct span (T3 in SRC's question, T4 in DST's), so this is a genuine **k+v transplant** (not query): cache the SRC **T3-description** key+value and transplant it onto the DST **T4-description** span. Test: does patched DST implement **T3** instead of T4? Gaze is **N/A** (T3 never appears in the DST prompt); eval = causal + readout.

In [7]:
# ALIGN_KNOB = "pad"
# # PATCH_HEADS = UNION_HEADS
# # ALPHA = 2.0
# PATCH_HEADS = [(layer_idx, head_idx) for layer_idx in range(35, 50) for head_idx in range(mt.n_heads)]
# ALPHA = 1.0
# len(PATCH_HEADS)

In [26]:
mp = D.multi_predicate_setups(
    "list[str]", n_conditions=4, n_samples=16, seed=0,
    src_lang="python", dst_lang="cpp", naming="letters",
)
for s in mp:
    print(f"  shared={[p.id for p in s.shared]}  SRC.frontier={s.expected_next:<16} DST.frontier={s.dst_next}")

inst_mp = mp[14]
mm = D.materialize(inst_mp)
MP_SRC0 = token_span(mm["src_prompt"], mm["src_task_text"])   # T3 description (pre-align)
MP_DST0 = token_span(mm["dst_prompt"], mm["dst_task_text"])   # T4 description (pre-align)
print(f"\nSRC frontier (T3) = {mm['src_task_text']!r}")
print(f"DST frontier (T4) = {mm['dst_task_text']!r}")
print(f"shared (T1,T2)    = {inst_mp.task_desc!r}")
mp_src_tok, mp_dst_tok, MP_SRC_POS, MP_DST_POS, MP_SRC_SPAN, MP_DST_SPAN = align_kv_spans(
    mt, mm["src_prompt"], mm["dst_prompt"], MP_SRC0, MP_DST0, mode=ALIGN_KNOB,
    src_text=mm["src_task_text"], dst_text=mm["dst_task_text"])
show_spans(mp_src_tok, f"SRC · T3 CACHED-FROM {tuple(MP_SRC_SPAN)} (orange) · pivot@{mp_src_tok['input_ids'].shape[1]-1} · align={ALIGN_KNOB}",
           task_span=MP_SRC_SPAN, kv_span=(MP_SRC_POS[0], MP_SRC_POS[-1] + 1), piv=-1)
show_spans(mp_dst_tok, f"DST · T4 PATCHED-TO {tuple(MP_DST_SPAN)} (orange) · pivot@{mp_dst_tok['input_ids'].shape[1]-1} · align={ALIGN_KNOB}",
           task_span=MP_DST_SPAN, kv_span=(MP_DST_POS[0], MP_DST_POS[-1] + 1), piv=-1)

  shared=['any_empty', 'total_uppercase_letters', 'count_with_digit']  SRC.frontier=has_palindrome   DST.frontier=distinct_count
  shared=['total_uppercase_letters', 'count_with_digit', 'any_empty']  SRC.frontier=distinct_count   DST.frontier=count_all_lowercase
  shared=['total_vowels', 'longest_word_length', 'count_capitalized']  SRC.frontier=count_short      DST.frontier=count_long
  shared=['longest_word_length', 'count_long', 'any_empty']  SRC.frontier=total_length     DST.frontier=total_uppercase_letters
  shared=['count_short', 'count_ends_consonant', 'total_uppercase_letters']  SRC.frontier=total_vowels     DST.frontier=longest_word_length
  shared=['distinct_count', 'total_uppercase_letters', 'total_length']  SRC.frontier=total_vowels     DST.frontier=all_unique_lengths
  shared=['shortest_word_length', 'count_all_lowercase', 'count_short']  SRC.frontier=total_length     DST.frontier=count_starts_vowel
  shared=['all_unique_lengths', 'count_starts_vowel', 'longest_word_length'

╭─ SRC · T3 CACHED-FROM (63, 114) (orange) · pivot@194 · align=pad ───────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the number of words that end with a consonant\n                                                             │
│ (2) the total number of characters across all words\n                                                           │
│ (3) the total number of uppercase letters across all words\n                                                    │
│ (4) the number of words that start with a vowel\n                                                               │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = sum(1 for w in words if w and w[-1].isalpha() and w[-1].lower() not in "aeiou")\n                       │
│     b = sum(len(w) for w in words)\n                                                                            │
│     c = sum(1 for w in words for c in w if c.isupper())\n                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ DST · T4 PATCHED-TO (64, 115) (orange) · pivot@290 · align=pad ────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a C++ program that implements a function `func` which, given a list of strings `words`, computes:\n       │
│ \n                                                                                                              │
│ (1) the number of words that end with a consonant\n                                                             │
│ (2) the total number of characters across all words\n                                                           │
│ (3) the total number of uppercase letters across all words\n                                                    │
│ (4) the number of words shorter than three characters\n                                                         │
│ Assume that all the libraries have been already imported.\n                                                     │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ auto func(std::vector<std::string> words){\n                                                                    │
│     int a = 0;\n                                                                                                │
│     for(auto &a_w : words){ if(!a_w.empty()){ char a_c = a_w.back(); if(std::isalpha((unsigned char)a_c)){ char │
│ a_l = std::tolower((unsigned char)a_c); if(!(a_l=='a'||a_l=='e'||a_l=='i'||a_l=='o'||a_l=='u')) a++; } } }\n    │
│     int b = 0;\n                                                                                                │
│     for (auto& w : words) b += w.size();\n                                                                      │
│     int c = 0;\n                                                                                                │
│     for(auto &c_w : words) for(char c_c : c_w) if(std::isupper((unsigned char)c_c)) c++;\n                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Multi-predicate intervention — k+v transplant of the frontier-predicate span

Same `run_kv_transfer_intervention` as the kv-cluster (align the T3/T4 spans per `ALIGN_KNOB`, cache SRC key+value over `PATCH_HEADS`, GQA-corrected headwise routing), scored with `eval_kv_transfer` (causal + readout; no gaze). `ALPHA` applies.

In [27]:
#------------------------------------------------------------------------------------------------------
ALPHA = 2.0        # OV-write boost of PATCH_HEADS at the pivot on the PATCHED gen/readout (1.0 = off)
ALIGN_KNOB = "pad"   # k/v transplant window (A4 knob): "right" = right-edge align (drop the longer
                       #   span's front) ; "pad" = left-pad the shorter span so the dest span is fully
                       #   overwritten (works cross-language too)
PATCH_HEADS = GOAL_HEADS                    # heads carrying the transplant (GOAL | UNION | ...)
READOUT_HEADS = GOAL_HEADS                  # heads read by goalscope (I3); kept SEPARATE from PATCH_HEADS
# ── ablation (I2): ZERO the OV write of these heads on the PATCHED run — kill the "backup" goal
# calc the KV-patch inconsistency triggers. Default = every head in ABLATE_LAYERS minus PATCH_HEADS.
ABLATE_LAYERS = range(33, 43)   # layers 35–50 (inclusive) — where GOAL_HEADS concentrate
ABLATE_HEADS = sorted({(l, h) for l in ABLATE_LAYERS for h in range(mt.n_heads)} - set(PATCH_HEADS))
# ABLATE_HEADS = []
ABLATION_STRAT = "restrict"   # how ABLATE_HEADS are ablated on the PATCHED run:
#  "zero"     -> kill their OV write (×0.0 at ALL positions) -- the original strategy
#  "restrict" -> keep their write but attention-knockout the <task_span>: they attend to every
#                context token EXCEPT the task-description columns (pre-softmax -inf)
#------------------------------------------------------------------------------------------------------

In [28]:
from src.diff.functional import run_kv_transfer_intervention
from src.diff.eval import eval_kv_transfer


def report(tag, inst, out):
    ev = eval_kv_transfer(
        inst,
        patched_gen=out["gen_patched"],
        readouts=(out["ro_src"], out["ro_dst"], out["ro_patched"]),
    )
    rich_print(
        f"[bold]{tag}[/bold]  SRC.target={inst.expected_next_desc!r}  DST.target={inst.dst_next_desc!r}"
    )
    rich_print(f"  [red1]clean SRC[/red1]    :\n{escape(out['gen_src'])}")
    rich_print(f"  [sea_green3]clean DST[/sea_green3]    :\n{escape(out['gen_dst'])}")
    rich_print(f"  [salmon1]patched DST[/salmon1]  :\n{escape(out['gen_patched'])}")
    rich_print(
        f"  readout src→dst→pat : {out['ro_src']!r} → {out['ro_dst']!r} → {out['ro_patched']!r}"
    )
    rich_print(f"  [bold]eval[/bold] : {ev.summary()}")
    return ev


# k+v transplant of the SRC frontier (T3) description onto the DST frontier (T4) span
report(
    f"multi-predicate · k+v transplant · {inst_mp.meta['datatype']} · align={ALIGN_KNOB}",
    inst_mp,
    run_kv_transfer_intervention(
        mt=mt,
        inst=inst_mp,
        patch_heads=PATCH_HEADS,
        readout_heads=READOUT_HEADS,
        alpha=ALPHA,
        align=ALIGN_KNOB,
        ablate_heads=ABLATE_HEADS,
        ablation_strat=ABLATION_STRAT,
    ),
)

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

pos=194 '    ' => I'm checking if the character is a vowel (a, e, i, o, u).

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

pos=290 '    ' => Counting lines where the length of the line is less than 1.

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

pos=290 '    ' => Counting lines where the condition `len(line)` is true.

... (3 outputs left out)

patched DST  :
 int d = 0;
    for(auto &d_w : words) if(!d_w.empty()){ char d_c = d_w.front(); if(std::isalpha((unsigned char)d_c)){ char d_l
= std::tolower((unsigned char)d_c); if(d_l=='a'||d_l=='e'||d_l=='i'||d_l=='o'||d_l=='u') d++; } }
    return std::make_tuple(a, b, c, d

readout src→dst→pat : "I'm checking if the character is a vowel (a, e, i, o, u)." → 'Counting lines where the 
length of the line is less than 1.' → 'Counting lines where the condition `len(line)` is true.'

eval : causal: TRIGGERED (src) | readout: src→neither→neither [INVALID, no-shift]

QTransferEval(causal={'label': 'src', 'why': 'The code checks the first character of each word against vowels (a, e, i, o, u), counting words that start with a vowel — that is sub-goal A.', 'raw': '<verdict>A</verdict>\n<why>The code checks the first character of each word against vowels (a, e, i, o, u), counting words that start with a vowel — that is sub-goal A.</why>', 'triggered': True}, readout={'source': 'src', 'dest': 'neither', 'patched': 'neither', 'valid': False, 'shifted': False, 'details': {'source': {'label': 'src', 'why': 'Checking if a character is a vowel is the core logic needed to determine if a word starts with a vowel (sub-goal A), not related to word length (sub-goal B).', 'raw': '<verdict>A</verdict><why>Checking if a character is a vowel is the core logic needed to determine if a word starts with a vowel (sub-goal A), not related to word length (sub-goal B).</why>'}, 'dest': {'label': 'neither', 'why': 'The computation checks if length < 1 (empty lines), which ma